# Лабораторная работа № 4 — массивы и JSON

Предварительно выполните предыдущую лабораторную. Создание таблиц рассчитано на первый запуск: при повторном запуске существующие таблицы не удаляются, PostgreSQL сообщит об их наличии. Все изменения сохраняются только в последней ячейке. Если прервали выполнение, выполните `conn.rollback()` и начните заново.

Выполняйте ячейки сверху вниз (**Run All**). Запросы находятся прямо в ноутбуке; SQL-файлы не читаются. Параметры подключения — в следующей ячейке. Зависимости: `pip install "psycopg[binary]" pandas ipykernel` (также поддерживается psycopg2).

[Все лабораторные](../../README.md) · [Отчёты ЛР4](../../labs/lab04/)

## Содержание

- [1. Учебные примеры: графики и увлечения](#part-1)
- [2. Индивидуальное задание: услуги и события проезда](#part-2)

In [1]:
import os
import getpass
import pandas as pd
from IPython.display import display
try:
    import psycopg
except ImportError:
    import psycopg2 as psycopg

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)
# При необходимости задайте PGHOST, PGPORT, PGUSER, PGPASSWORD перед запуском.
# Пароль в ноутбуке не сохраняйте: драйвер также поддерживает ~/.pgpass.
DB_NAME = os.getenv("PGDATABASE", "parking")
conn = psycopg.connect(
    host=os.getenv("PGHOST", "127.0.0.1"),
    port=int(os.getenv("PGPORT", "5432")),
    user=os.getenv("PGUSER", getpass.getuser()),
    password=os.getenv("PGPASSWORD") or None,
    dbname=DB_NAME,
)
conn.autocommit = False

def q(sql):
    """Выполнить инструкцию; выборку показать таблицей."""
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
            if cur.description:
                frame = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
                frame.index = range(1, len(frame) + 1)
                display(frame)
                print(f"Строк: {len(frame)}")
            else:
                print(cur.statusmessage)
    except Exception:
        conn.rollback()
        raise

run = q
print("Подключение:", DB_NAME)


Подключение: parking


<a id="part-1"></a>

## 1. Учебные примеры: графики и увлечения

Лабораторная работа № 4, п. 1 — учебный пример из пособия (стр. 91–100)

«Работа с массивами и типами JSON»

Выполнять, подключившись к базе данных parking

Запросы повторяют пример пособия; вместо сотрудников базы sales

используются сотрудники таблицы employees базы parking (id 1–4).

1. МАССИВЫ: графики работы сотрудников (таблица schedules)

In [2]:
q("""
CREATE TABLE schedules (
    sched_ID     integer NOT NULL,
    sched_emp_ID integer NOT NULL,
    schedule     integer[],
    CONSTRAINT schedules_PK PRIMARY KEY (sched_ID)
);
""")

CREATE TABLE


In [3]:
q("""
ALTER TABLE schedules
ADD CONSTRAINT schedules_FK FOREIGN KEY (sched_emp_ID)
REFERENCES employees (emp_ID)
ON DELETE CASCADE
ON UPDATE CASCADE;
""")

ALTER TABLE


Четыре графика (по одному для каждого сотрудника): номера дней недели

In [4]:
q("""
INSERT INTO schedules VALUES
    (1, 1, '{ 1, 3, 5, 6, 7 }'::integer[]),
    (2, 2, '{ 1, 2, 5, 7 }'::integer[]),
    (3, 3, '{ 2, 5 }'::integer[]),
    (4, 4, '{ 3, 5, 6 }'::integer[]);
""")

INSERT 0 4


In [5]:
q("""
SELECT schedules.sched_ID, employees.emp_name, schedules.schedule
FROM schedules JOIN employees
ON schedules.sched_emp_ID = employees.emp_ID;
""")

,sched_id,emp_name,schedule
1,1,Гордеев Пётр Ильич,"[1, 3, 5, 6, 7]"
2,2,Сомова Анна Львовна,"[1, 2, 5, 7]"
3,3,Ким Дмитрий Сергеевич,"[2, 5]"
4,4,Ветров Илья Романович,"[3, 5, 6]"


Строк: 4


Каждый сотрудник должен работать 4 раза в неделю

Сотруднику 4: добавить один день операцией конкатенации

In [6]:
q("""
UPDATE schedules
SET schedule = schedule || 7
WHERE sched_ID = 4;
""")

UPDATE 1


Сотруднику 3: добавить день в конец массива (array_append)

In [7]:
q("""
UPDATE schedules
SET schedule = array_append(schedule, 6)
WHERE sched_ID = 3;
""")

UPDATE 1


Сотруднику 3: добавить день в начало массива (array_prepend,

параметры функции поменялись местами)

In [8]:
q("""
UPDATE schedules
SET schedule = array_prepend(1, schedule)
WHERE sched_ID = 3;
""")

UPDATE 1


Сотруднику 1: удалить из графика пятницу (array_remove)

In [9]:
q("""
UPDATE schedules
SET schedule = array_remove(schedule, 5)
WHERE sched_ID = 1;
""")

UPDATE 1


Сотруднику 2: изменить дни работы по индексам элементов

(нумерация с единицы)

In [10]:
q("""
UPDATE schedules
SET schedule[1] = 2, schedule[2] = 3
WHERE sched_ID = 2;
""")

UPDATE 1


То же самое срезом массива (slice); запрос возвращает прежний результат

In [11]:
q("""
UPDATE schedules
SET schedule[1:2] = ARRAY[2, 3]
WHERE sched_ID = 2;
""")

UPDATE 1


In [12]:
q("""
SELECT * FROM schedules;
""")

,sched_id,sched_emp_id,schedule
1,4,4,"[3, 5, 6, 7]"
2,3,3,"[1, 2, 5, 6]"
3,1,1,"[1, 3, 6, 7]"
4,2,2,"[2, 3, 5, 7]"


Строк: 4


Выборки из таблицы с массивами

Сотрудники, работающие в среду (array_position — индекс первого

вхождения элемента или NULL)

In [13]:
q("""
SELECT * FROM schedules
WHERE array_position(schedule, 3) IS NOT NULL;
""")

,sched_id,sched_emp_id,schedule
1,4,4,"[3, 5, 6, 7]"
2,1,1,"[1, 3, 6, 7]"
3,2,2,"[2, 3, 5, 7]"


Строк: 3


Сотрудники, работающие по понедельникам и воскресеньям

(@> — левый массив содержит все элементы правого)

In [14]:
q("""
SELECT * FROM schedules
WHERE schedule @> '{1, 7}'::integer[];
""")

,sched_id,sched_emp_id,schedule
1,1,1,"[1, 3, 6, 7]"


Строк: 1


Работающие по вторникам и/или пятницам

(&& — у массивов есть общие элементы)

In [15]:
q("""
SELECT * FROM schedules
WHERE schedule && ARRAY[2, 5];
""")

,sched_id,sched_emp_id,schedule
1,4,4,"[3, 5, 6, 7]"
2,3,3,"[1, 2, 5, 6]"
3,2,2,"[2, 3, 5, 7]"


Строк: 3


Не работающие ни во вторник, ни в пятницу

In [16]:
q("""
SELECT * FROM schedules
WHERE NOT (schedule && ARRAY[2, 5]);
""")

,sched_id,sched_emp_id,schedule
1,1,1,"[1, 3, 6, 7]"


Строк: 1


Развернуть массив в столбец таблицы (unnest)

In [17]:
q("""
SELECT unnest(schedule) AS день_недели
FROM schedules
WHERE sched_emp_ID = 1;
""")

,день_недели
1,1
2,3
3,6
4,7


Строк: 4


2. JSON: увлечения сотрудников (таблица hobbies, связь «один-к-одному»)

In [18]:
q("""
CREATE TABLE hobbies (
    hobb_emp_ID integer NOT NULL,
    hobbies     jsonb,
    CONSTRAINT hobbies_PK PRIMARY KEY (hobb_emp_ID)
);
""")

CREATE TABLE


In [19]:
q("""
ALTER TABLE hobbies
ADD CONSTRAINT hobbies_FK FOREIGN KEY (hobb_emp_ID)
REFERENCES employees (emp_ID)
ON DELETE CASCADE
ON UPDATE CASCADE;
""")

ALTER TABLE


In [20]:
q("""
INSERT INTO hobbies VALUES
    (1, '{ "sports": ["волейбол", "плавание"],
           "home_lib": true, "trips": 3
         }'::jsonb ),
    (2, '{ "sports": ["теннис", "плавание"],
           "home_lib": true, "trips": 2
         }'::jsonb ),
    (3, '{ "sports": ["плавание"],
           "home_lib": false, "trips": 4
         }'::jsonb ),
    (4, '{ "sports": ["волейбол", "плавание", "теннис"],
           "home_lib": true, "trips": 0
         }'::jsonb );
""")

INSERT 0 4


In [21]:
q("""
SELECT * FROM hobbies;
""")

,hobb_emp_id,hobbies
1,1,"{'trips': 3, 'sports': ['волейбол', 'плавание'], 'home_lib': True}"
2,2,"{'trips': 2, 'sports': ['теннис', 'плавание'], 'home_lib': True}"
3,3,"{'trips': 4, 'sports': ['плавание'], 'home_lib': False}"
4,4,"{'trips': 0, 'sports': ['волейбол', 'плавание', 'теннис'], 'home_lib': True}"


Строк: 4


Выборки из таблицы с JSON

Все, кто занимается волейболом

In [22]:
q("""
SELECT * FROM hobbies
WHERE hobbies @> '{"sports": ["волейбол"]}'::jsonb;
""")

,hobb_emp_id,hobbies
1,1,"{'trips': 3, 'sports': ['волейбол', 'плавание'], 'home_lib': True}"
2,4,"{'trips': 0, 'sports': ['волейбол', 'плавание', 'теннис'], 'home_lib': True}"


Строк: 2


То же, но выводятся только виды спорта (-> — доступ к ключу)

In [23]:
q("""
SELECT hobb_emp_ID, hobbies -> 'sports' AS sports
FROM hobbies
WHERE hobbies -> 'sports' @> '[ "волейбол" ]'::jsonb;
""")

,hobb_emp_id,sports
1,1,"[волейбол, плавание]"
2,4,"[волейбол, плавание, теннис]"


Строк: 2


Проверка наличия ключа (?): ключа «sport» нет — результат 0

In [24]:
q("""
SELECT count( * )
FROM hobbies
WHERE hobbies ? 'sport';
""")

,count
1,0


Строк: 1


Ключ «sports» есть во всех строках — результат 4

In [25]:
q("""
SELECT count( * )
FROM hobbies
WHERE hobbies ? 'sports';
""")

,count
1,4


Строк: 1


Обновление JSON-объектов

Сотрудник 4 решил заниматься только шахматами

In [26]:
q("""
UPDATE hobbies
SET hobbies = hobbies || '{"sports": ["шахматы"]}'
WHERE hobb_emp_ID = 4;
""")

UPDATE 1


In [27]:
q("""
SELECT hobb_emp_ID, hobbies
FROM hobbies
WHERE hobb_emp_ID = 4;
""")

,hobb_emp_id,hobbies
1,4,"{'trips': 0, 'sports': ['шахматы'], 'home_lib': True}"


Строк: 1


Сотрудник 4 возобновил занятия плаванием: jsonb_set добавляет

элемент с номером 1 (нумерация с нуля) в массив sports

In [28]:
q("""
UPDATE hobbies
SET hobbies = jsonb_set(hobbies, '{ sports, 1 }','"плавание"')
WHERE hobb_emp_ID = 4;
""")

UPDATE 1


In [29]:
q("""
SELECT hobb_emp_ID, hobbies
FROM hobbies
WHERE hobb_emp_ID = 4;
""")

,hobb_emp_id,hobbies
1,4,"{'trips': 0, 'sports': ['шахматы', 'плавание'], 'home_lib': True}"


Строк: 1


<a id="part-2"></a>

## 2. Индивидуальное задание: услуги и события проезда

Лабораторная работа № 4 (РГР, вариант 11 — «Автостоянка»)

Столбцы с массивами и данными в формате JSON

Выполнять, подключившись к базе данных parking, после lab3_parking_data.sql

Содержание файла:

1. Таблица со столбцом-массивом   (contract_options, text[])

2. Таблица со столбцом JSON       (passage_events, jsonb)

3. Заполнение таблиц данными      (INSERT)

4. Обработка данных в массивах

5. Обработка данных в JSON

1. ТАБЛИЦА СО СТОЛБЦОМ-МАССИВОМ

Дополнительные услуги по договору хранятся массивом строк, а не

отдельной таблицей «многие-ко-многим»: набор услуг невелик,

не имеет собственных атрибутов и всегда читается вместе с договором.

Связь с договором — «один-к-одному» (UNIQUE на внешний ключ).

In [30]:
q("""
CREATE TABLE contract_options (
    opt_ID       integer NOT NULL,
    opt_contr_ID integer NOT NULL,
    opt_services text[]  NOT NULL DEFAULT '{}',
    CONSTRAINT contract_options_PK PRIMARY KEY (opt_ID),
    CONSTRAINT contract_options_UK UNIQUE (opt_contr_ID),
    CONSTRAINT contract_options_FK FOREIGN KEY (opt_contr_ID)
        REFERENCES contracts (contr_ID) ON DELETE CASCADE ON UPDATE CASCADE
);
""")

CREATE TABLE


In [31]:
q("""
COMMENT ON TABLE contract_options IS
    'Дополнительные услуги по договору хранения (столбец-массив text[])';
""")

COMMENT


2. ТАБЛИЦА СО СТОЛБЦОМ JSON

Камера у шлагбаума присылает набор полей, состав которого меняется

(варианты распознавания, освещённость, скорость и т. д.), поэтому

«сырой» ответ хранится документом jsonb. Тип jsonb (а не json)

хранится в разобранном виде, поддерживает операторы @>, ?, ||, -

и индексы.

In [32]:
q("""
CREATE TABLE passage_events (
    ev_ID      integer NOT NULL,
    ev_pass_ID integer NOT NULL,
    ev_payload jsonb   NOT NULL,
    CONSTRAINT passage_events_PK PRIMARY KEY (ev_ID),
    CONSTRAINT passage_events_UK UNIQUE (ev_pass_ID),
    CONSTRAINT passage_events_FK FOREIGN KEY (ev_pass_ID)
        REFERENCES passages (pass_ID) ON DELETE CASCADE ON UPDATE CASCADE
);
""")

CREATE TABLE


In [33]:
q("""
COMMENT ON TABLE passage_events IS
    'Данные камеры по событию проезда (столбец JSON, тип jsonb)';
""")

COMMENT


3. ЗАПОЛНЕНИЕ ТАБЛИЦ ДАННЫМИ

Дополнительные услуги по договорам (два способа записи массива:

конструктор ARRAY[...] и текстовый литерал '{...}')

In [34]:
q("""
INSERT INTO contract_options (opt_ID, opt_contr_ID, opt_services)
     VALUES ( 1,  1, ARRAY['мойка', 'охрана']),
            ( 2,  2, ARRAY['мойка']),
            ( 3,  3, ARRAY['подогрев', 'охрана', 'мойка']),
            ( 4,  4, '{}'),
            ( 5,  5, '{"охрана"}'),
            ( 6,  6, '{"подогрев","шиномонтаж"}'),
            ( 7,  7, ARRAY['охрана']),
            ( 8,  8, '{}'),
            ( 9,  9, ARRAY['мойка', 'зарядка электромобиля']),
            (10, 10, '{}');
""")

INSERT 0 10


Данные камер по проездам (в том числе по событиям с низкой

достоверностью — для них массив вариантов содержит несколько

кандидатов, а для проезда без вариантов — пустой массив)

In [35]:
q("""
INSERT INTO passage_events (ev_ID, ev_pass_ID, ev_payload)
     VALUES ( 1,  1, '{"camera":"CAM-1","lighting":"day","speed_kmh":8,
                       "candidates":[{"plate":"А123ВС55","conf":0.981}]}'),
            ( 2,  2, '{"camera":"CAM-2","lighting":"evening","speed_kmh":11,
                       "candidates":[{"plate":"А123ВС55","conf":0.975}]}'),
            ( 3,  3, '{"camera":"CAM-1","lighting":"day","speed_kmh":9,
                       "candidates":[{"plate":"В456ЕК55","conf":0.964}]}'),
            ( 4,  5, '{"camera":"CAM-1","lighting":"day","speed_kmh":14,
                       "candidates":[{"plate":"Т777ТТ77","conf":0.642},
                                     {"plate":"Т777ТТ11","conf":0.311}]}'),
            ( 5,  7, '{"camera":"CAM-3","lighting":"dawn","speed_kmh":6,
                       "candidates":[{"plate":"Е789МН55","conf":0.933}]}'),
            ( 6, 10, '{"camera":"CAM-2","lighting":"night","speed_kmh":10,
                       "candidates":[{"plate":"К321ОР55","conf":0.979}]}'),
            ( 7, 13, '{"camera":"CAM-3","lighting":"dawn","speed_kmh":7,
                       "candidates":[{"plate":"Н987УХ55","conf":0.902},
                                     {"plate":"Н987УХ35","conf":0.288}]}'),
            ( 8, 16, '{"camera":"CAM-2","lighting":"night","speed_kmh":18,
                       "candidates":[]}'),
            ( 9, 17, '{"camera":"CAM-1","lighting":"day","speed_kmh":5,
                       "candidates":[{"plate":"С369МН155","conf":0.968}]}'),
            (10, 19, '{"camera":"CAM-1","lighting":"day","speed_kmh":8,
                       "candidates":[{"plate":"А123ВС55","conf":0.984}]}');
""")

INSERT 0 10


In [36]:
q("""
SELECT 'contract_options' AS "Таблица", count(*) AS "Строк" FROM contract_options
UNION ALL SELECT 'passage_events', count(*) FROM passage_events;
""")

,Таблица,Строк
1,contract_options,10
2,passage_events,10


Строк: 2


4. ОБРАБОТКА ДАННЫХ В СТОЛБЦЕ-МАССИВЕ

4.1. Выборка договоров вместе с массивом услуг

и числом услуг в нём

In [37]:
q("""
SELECT c.contr_number   AS "Договор",
       o.opt_services   AS "Услуги",
       coalesce(array_length(o.opt_services, 1), 0) AS "Число услуг"
  FROM contract_options o
  JOIN contracts c ON c.contr_ID = o.opt_contr_ID
 ORDER BY c.contr_ID;
""")

,Договор,Услуги,Число услуг
1,Д-2024/001,"[мойка, охрана]",2
2,Д-2024/002,[мойка],1
3,Д-2024/003,"[подогрев, охрана, мойка]",3
4,Д-2024/004,[],0
5,Д-2024/005,[охрана],1
6,Д-2024/006,"[подогрев, шиномонтаж]",2
7,Д-2024/007,[охрана],1
8,Д-2024/008,[],0
9,Д-2024/009,"[мойка, зарядка электромобиля]",2
10,Д-2024/010,[],0


Строк: 10


4.2. Доступ к элементу по индексу (нумерация с 1)

In [38]:
q("""
SELECT opt_contr_ID AS "Договор",
       opt_services[1] AS "Первая услуга"
  FROM contract_options
 WHERE opt_services[1] IS NOT NULL;
""")

,Договор,Первая услуга
1,1,мойка
2,2,мойка
3,3,подогрев
4,5,охрана
5,6,подогрев
6,7,охрана
7,9,мойка


Строк: 7


4.3. Поиск: договоры, в которых есть услуга «мойка»

(оператор ANY)

In [39]:
q("""
SELECT opt_contr_ID AS "Договор", opt_services AS "Услуги"
  FROM contract_options
 WHERE 'мойка' = ANY (opt_services);
""")

,Договор,Услуги
1,1,"[мойка, охрана]"
2,2,[мойка]
3,3,"[подогрев, охрана, мойка]"
4,9,"[мойка, зарядка электромобиля]"


Строк: 4


4.4. Поиск: договоры, содержащие обе услуги «охрана» и «мойка»

(оператор содержания @>)

In [40]:
q("""
SELECT opt_contr_ID AS "Договор", opt_services AS "Услуги"
  FROM contract_options
 WHERE opt_services @> ARRAY['охрана', 'мойка'];
""")

,Договор,Услуги
1,1,"[мойка, охрана]"
2,3,"[подогрев, охрана, мойка]"


Строк: 2


4.5. Развёртывание массива в строки (unnest) и подсчёт,

сколько договоров заказали каждую услугу

In [41]:
q("""
SELECT s AS "Услуга", count(*) AS "Договоров"
  FROM contract_options, unnest(opt_services) AS s
 GROUP BY s
 ORDER BY count(*) DESC, s;
""")

,Услуга,Договоров
1,мойка,4
2,охрана,4
3,подогрев,2
4,зарядка электромобиля,1
5,шиномонтаж,1


Строк: 5


4.6. Добавление элемента в массив (array_append)

In [42]:
q("""
UPDATE contract_options
   SET opt_services = array_append(opt_services, 'охрана')
 WHERE opt_contr_ID = 4;
""")

UPDATE 1


4.7. Удаление элемента из массива (array_remove)

In [43]:
q("""
UPDATE contract_options
   SET opt_services = array_remove(opt_services, 'мойка')
 WHERE opt_contr_ID = 3;
""")

UPDATE 1


4.8. Замена массива целиком и объединение массивов (||)

In [44]:
q("""
UPDATE contract_options
   SET opt_services = opt_services || ARRAY['мойка', 'подогрев']
 WHERE opt_contr_ID = 8;
""")

UPDATE 1


In [45]:
q("""
SELECT opt_contr_ID AS "Договор", opt_services AS "Услуги после изменений"
  FROM contract_options
 WHERE opt_contr_ID IN (3, 4, 8)
 ORDER BY opt_contr_ID;
""")

,Договор,Услуги после изменений
1,3,"[подогрев, охрана]"
2,4,[охрана]
3,8,"[мойка, подогрев]"


Строк: 3


5. ОБРАБОТКА ДАННЫХ В СТОЛБЦЕ JSON

5.1. Извлечение значений: -> возвращает JSON, ->> возвращает текст

In [46]:
q("""
SELECT ev_pass_ID                           AS "Проезд",
       ev_payload ->> 'camera'              AS "Камера",
       ev_payload ->> 'lighting'            AS "Освещение",
       (ev_payload ->> 'speed_kmh')::int    AS "Скорость, км/ч"
  FROM passage_events
 ORDER BY ev_pass_ID;
""")

,Проезд,Камера,Освещение,"Скорость, км/ч"
1,1,CAM-1,day,8
2,2,CAM-2,evening,11
3,3,CAM-1,day,9
4,5,CAM-1,day,14
5,7,CAM-3,dawn,6
6,10,CAM-2,night,10
7,13,CAM-3,dawn,7
8,16,CAM-2,night,18
9,17,CAM-1,day,5
10,19,CAM-1,day,8


Строк: 10


5.2. Доступ к вложенным данным: первый вариант распознавания

номера (элемент массива candidates по индексу, нумерация с 0)

In [47]:
q("""
SELECT ev_pass_ID AS "Проезд",
       ev_payload -> 'candidates' -> 0 ->> 'plate'              AS "Лучший вариант",
       (ev_payload -> 'candidates' -> 0 ->> 'conf')::numeric    AS "Достоверность"
  FROM passage_events
 ORDER BY ev_pass_ID;
""")

,Проезд,Лучший вариант,Достоверность
1,1,А123ВС55,0.981
2,2,А123ВС55,0.975
3,3,В456ЕК55,0.964
4,5,Т777ТТ77,0.642
5,7,Е789МН55,0.933
6,10,К321ОР55,0.979
7,13,Н987УХ55,0.902
8,16,NaN,None
9,17,С369МН155,0.968
10,19,А123ВС55,0.984


Строк: 10


5.3. Фильтрация по содержимому документа

(оператор @> — «JSON содержит фрагмент»)

In [48]:
q("""
SELECT ev_pass_ID AS "Проезд", ev_payload ->> 'camera' AS "Камера"
  FROM passage_events
 WHERE ev_payload @> '{"lighting":"night"}';
""")

,Проезд,Камера
1,10,CAM-2
2,16,CAM-2


Строк: 2


5.4. Фильтрация по числовому значению внутри JSON:

проезды со скоростью выше 10 км/ч

In [49]:
q("""
SELECT ev_pass_ID AS "Проезд", (ev_payload ->> 'speed_kmh')::int AS "Скорость, км/ч"
  FROM passage_events
 WHERE (ev_payload ->> 'speed_kmh')::int > 10;
""")

,Проезд,"Скорость, км/ч"
1,2,11
2,5,14
3,16,18


Строк: 3


5.5. Развёртывание вложенного массива в строки

(jsonb_array_elements): все варианты распознавания

In [50]:
q("""
SELECT e.ev_pass_ID          AS "Проезд",
       c ->> 'plate'         AS "Вариант номера",
       (c ->> 'conf')::numeric AS "Достоверность"
  FROM passage_events e,
       jsonb_array_elements(e.ev_payload -> 'candidates') AS c
 ORDER BY e.ev_pass_ID, (c ->> 'conf')::numeric DESC;
""")

,Проезд,Вариант номера,Достоверность
1,1,А123ВС55,0.981
2,2,А123ВС55,0.975
3,3,В456ЕК55,0.964
4,5,Т777ТТ77,0.642
5,5,Т777ТТ11,0.311
6,7,Е789МН55,0.933
7,10,К321ОР55,0.979
8,13,Н987УХ55,0.902
9,13,Н987УХ35,0.288
10,17,С369МН155,0.968


Строк: 11


5.6. Агрегирование по полю JSON: число проездов по камерам

In [51]:
q("""
SELECT ev_payload ->> 'camera' AS "Камера", count(*) AS "Проездов"
  FROM passage_events
 GROUP BY ev_payload ->> 'camera'
 ORDER BY 1;
""")

,Камера,Проездов
1,CAM-1,5
2,CAM-2,3
3,CAM-3,2


Строк: 3


5.7. Изменение значения внутри JSON (jsonb_set)

In [52]:
q("""
UPDATE passage_events
   SET ev_payload = jsonb_set(ev_payload, '{speed_kmh}', '10')
 WHERE ev_pass_ID = 1;
""")

UPDATE 1


5.8. Добавление нового ключа (оператор || объединяет документы)

In [53]:
q("""
UPDATE passage_events
   SET ev_payload = ev_payload || '{"verified":true}'
 WHERE (ev_payload -> 'candidates' -> 0 ->> 'conf')::numeric >= 0.9;
""")

UPDATE 8


5.9. Удаление ключа из документа (оператор -)

In [54]:
q("""
UPDATE passage_events
   SET ev_payload = ev_payload - 'lighting'
 WHERE ev_pass_ID = 19;
""")

UPDATE 1


In [55]:
q("""
SELECT ev_pass_ID AS "Проезд", ev_payload AS "Документ после изменений"
  FROM passage_events
 WHERE ev_pass_ID IN (1, 5, 19)
 ORDER BY ev_pass_ID;
""")

,Проезд,Документ после изменений
1,1,"{'camera': 'CAM-1', 'lighting': 'day', 'verified': True, 'speed_kmh': 10, 'candidates': [{'conf': 0.981, 'plate': 'А123ВС55'}]}"
2,5,"{'camera': 'CAM-1', 'lighting': 'day', 'speed_kmh': 14, 'candidates': [{'conf': 0.642, 'plate': 'Т777ТТ77'}, {'conf': 0.311, 'plate': 'Т777ТТ11'}]}"
3,19,"{'camera': 'CAM-1', 'verified': True, 'speed_kmh': 8, 'candidates': [{'conf': 0.984, 'plate': 'А123ВС55'}]}"


Строк: 3


## Завершение

Выполните эту ячейку, чтобы завершить транзакцию и закрыть подключение. При ошибке изменения текущей транзакции отменяются; повторите ноутбук с начала.

In [56]:
conn.commit()
conn.close()
print("Работа завершена, изменения сохранены.")

Работа завершена, изменения сохранены.
